# Experiment 3: OpenEnded Speech Corpus Verification & Acoustic Processing
## Intelligent Multimodal Mock Interview Assessment System

### Objectives:
1. **Partition & Disjoint Verification**:
   - Verify that `gemini2_train.csv` (train), `gemini2_dev.csv` (validation), and `human_test.csv` (test) have zero overlapping audio files.
2. **Audio Availability Audit**:
   - Document that the 6.71 GB `wav.zip` archive was intentionally omitted to preserve storage.
   - Avoid deceptive claims of audio neural net training when WAV waveforms are not on disk.
3. **Candidate Acoustic Feature Pipeline**:
   - Test and verify `CandidateAudioAnalyzer` using `librosa` for live/uploaded mock interview candidate response analysis (MFCCs, Pitch tracking, RMS energy, and pause ratios).

In [ ]:
import os, sys
import pandas as pd

BASE = os.path.join("..", "dataset", "openended")
train_df = pd.read_csv(os.path.join(BASE, "gemini2_train.csv"))
dev_df   = pd.read_csv(os.path.join(BASE, "gemini2_dev.csv"))
test_df  = pd.read_csv(os.path.join(BASE, "human_test.csv"))
meta_df  = pd.read_csv(os.path.join(BASE, "meta_data.csv"))

s_train = set(train_df["wavname"])
s_dev   = set(dev_df["wavname"])
s_test  = set(test_df["wavname"])

print(f"Train recordings (Gemini): {len(s_train)}")
print(f"Dev recordings (Gemini):   {len(s_dev)}")
print(f"Test recordings (Human):   {len(s_test)}")
print(f"Overlap Train & Dev:       {len(s_train.intersection(s_dev))}")
print(f"Overlap Train & Test:      {len(s_train.intersection(s_test))}")
print(f"Overlap Dev & Test:        {len(s_dev.intersection(s_test))}")

## 2. Acoustic Feature Extraction Pipeline (Librosa)

In [ ]:
sys.path.append(os.path.join("..", "src"))
from audio_feature_extractor import CandidateAudioAnalyzer
import numpy as np

# Create 2.5s sample wave to verify extraction
sr = 16000
t = np.linspace(0, 2.5, int(sr * 2.5), endpoint=False)
y_sample = 0.4 * np.sin(2 * np.pi * 180 * t)  # 180 Hz voice tone
y_sample[int(sr * 1.0):int(sr * 1.3)] = 0.0001  # pause

analyzer = CandidateAudioAnalyzer(target_sr=sr)
features = analyzer.extract_features_from_array(y_sample, sr=sr)
for k, v in features.items():
    if k != "mfcc_mean":
        print(f"{k:<25s}: {v}")